# GTSAM Double-Difference RTK with integer ambiguity resolution

This notebook walks through **gtsam-first RTK** step by step: cssrlib provides the
GNSS observation front-end, GTSAM's `DoubleDifference{Pseudorange,CarrierPhase}Factor`
(from the inuex35/gtsam fork) form the rover-base double differences and estimate a
static rover position + float ambiguities with **incremental ISAM2**, and finally the
integers are resolved with cssrlib's LAMBDA (`resamb_lambda`).

Pipeline: **load RINEX -> per-epoch DD measurements -> reference satellite ->
build the float factor graph (ISAM2) -> integer AR -> evaluate**. The float graph
is built first; the **integer ambiguity resolution is unfolded explicitly at the
end** (Section 5) so every step is visible. Runs on the data bundled with cssrlib
(`src/cssrlib/data`).

In [1]:
import os
import numpy as np

import cssrlib.rinex as rn
import cssrlib.gnss as gn
from cssrlib.gnss import rSigRnx, uTYP, sat2prn
from cssrlib.rtk import rtkpos
import gtsam
from gtsam import symbol

SYSS = (gn.uGNSS.GPS, gn.uGNSS.GAL)        # constellations to use
X = symbol('x', 0)                         # static rover ECEF position node
def AM(sat, f): return symbol('n', int(sat) * 10 + f)  # SD ambiguity node

## 1. Load RINEX (rover, base, navigation)

`rtkpos` is the cssrlib double-difference engine. We seed `nav.x[0:3]` with the
approximate rover position so its `qcedit` can compute satellite elevations.
`xyz_ref` is the surveyed marker used only to score accuracy.

In [2]:
bdir = os.path.join(os.path.dirname(os.getcwd()), 'src', 'cssrlib', 'data') + os.sep
xyz_ref = np.array([-3962108.673, 3381309.574, 3668678.638])
pos_ref = gn.ecef2pos(xyz_ref)

sigs = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("GL1C"), rSigRnx("GL2W"),
        rSigRnx("GS1C"), rSigRnx("GS2W"),
        rSigRnx("EC1C"), rSigRnx("EC5Q"), rSigRnx("EL1C"), rSigRnx("EL5Q"),
        rSigRnx("ES1C"), rSigRnx("ES5Q")]
sigsb = [rSigRnx("GC1C"), rSigRnx("GC2W"), rSigRnx("GL1C"), rSigRnx("GL2W"),
         rSigRnx("GS1C"), rSigRnx("GS2W"),
         rSigRnx("EC1X"), rSigRnx("EC5X"), rSigRnx("EL1X"), rSigRnx("EL5X"),
         rSigRnx("ES1X"), rSigRnx("ES5X")]

dec = rn.rnxdec(); dec.setSignals(sigs)
nav = gn.Nav(); dec.decode_nav(bdir + 'SEPT078M.21P', nav)
decb = rn.rnxdec(); decb.setSignals(sigsb)
decb.decode_obsh(bdir + '3034078M1.21O'); dec.decode_obsh(bdir + 'SEPT078M1.21O')
nav.rb = [-3959400.631, 3385704.533, 3667523.111]    # base station ECEF
rb = np.array(nav.rb)
rtk = rtkpos(nav, dec.pos)
nav.x[0:3] = np.array(dec.pos)                        # seed for elevations
nf = nav.nf
print('frequencies per constellation:', nf, ' base-rover baseline:',
      round(np.linalg.norm(xyz_ref - rb) / 1e3, 2), 'km')

frequencies per constellation: 2  base-rover baseline: 5.29 km


## 2. Front-end: per-epoch double-difference measurements

`prepare_double_difference_measurements` returns, per epoch, the rover/base
satellite positions (`rs`/`rsb`), the common-satellite indices (`iu`/`ir`),
the common satellites (`sat`) and rover elevations (`el`). No EKF -- this is just
the geometry/observation bundle that the GTSAM factors consume.

In [3]:
frames = []
sync = rn.sync_obs_hold(dec, decb, maxage=nav.maxtdiff)
for ne, (obs, obsb, dt) in enumerate(sync):
    if ne >= 60:
        break
    if obsb is None:
        continue
    dd = rtk.prepare_double_difference_measurements(obs, obsb, pos_pred=dec.pos)
    if dd is not None:
        frames.append((obs, obsb, dd))

obs0, obsb0, dd0 = frames[0]
print(f'{len(frames)} epochs collected')
print('epoch 0 common sats:', [int(s) for s in dd0.sat])
print('epoch 0 elevations [deg]:', np.round(np.rad2deg(dd0.el), 1))

59 epochs collected
epoch 0 common sats: [3, 4, 6, 9, 14, 17, 19, 22, 28, 35, 39, 40, 45, 47, 53, 58]
epoch 0 elevations [deg]: [40.8 35.7 40.9 33.  25.2 85.4 61.6 16.  32.1 32.8 17.9 48.6 60.9 41.4
 27.8 18.7]


## 3. Reference satellite per constellation (gauge)

Only between-satellite double differences are observable, so one single-difference
ambiguity per constellation is unobservable. We pick the highest-elevation
satellite as the reference and later pin its ambiguity (any value works -- the DDs
are gauge-independent).

In [4]:
el_cum = {}
for (_, _, dd) in frames:
    for k, s in enumerate(dd.sat):
        if dd.el[k] > 0:
            el_cum[int(s)] = el_cum.get(int(s), 0.0) + dd.el[k]
ref_of = {}
for s, e in el_cum.items():
    sys = sat2prn(s)[0]
    if sys in SYSS and (sys not in ref_of or e > el_cum[ref_of[sys]]):
        ref_of[sys] = s
print('reference satellite per constellation:',
      {int(k): int(v) for k, v in ref_of.items()})

reference satellite per constellation: {0: 17, 1: 45}


## 4. Build the float factor graph incrementally (ISAM2)

For every epoch and every frequency we add, per target satellite:
a `DoubleDifferencePseudorangeFactor` and a `DoubleDifferenceCarrierPhaseFactor`
(both form the rover-base double difference internally via Sagnac-corrected
`gnss::geodist`). The carrier factor ties the reference and target
**between-receiver SD ambiguities**; the reference ambiguity is gauge-pinned to its
carrier-minus-code value (non-zero so cssrlib's `ddidx` can pivot on it).

ISAM2 uses **QR** factorization, whose joint marginals are robust. This loop builds
only the **float** solution -- integer resolution comes afterwards in Section 5.

In [5]:
params = gtsam.ISAM2Params(); params.setFactorization('QR')
isam = gtsam.ISAM2(params)
seen_am, pinned = set(), set()
nfac = 0

for ei, (obs, obsb, dd) in enumerate(frames):
    graph = gtsam.NonlinearFactorGraph()
    val = gtsam.Values()
    if ei == 0:
        val.insert(X, gtsam.Point3(*dec.pos))
        graph.add(gtsam.PriorFactorPoint3(
            X, gtsam.Point3(*dec.pos), gtsam.noiseModel.Isotropic.Sigma(3, 30.0)))

    by_sys = {}
    for k, s in enumerate(dd.sat):
        by_sys.setdefault(sat2prn(int(s))[0], []).append(k)
    for sys, ks in by_sys.items():
        ref = ref_of.get(sys)
        ridx = next((k for k in ks if int(dd.sat[k]) == ref), None)
        if ridx is None:
            continue
        for f in range(nf):
            lam = obs.sig[sys][uTYP.L][f].wavelength()
            pr_rr, pr_br = obs.P[dd.iu[ridx], f], obsb.P[dd.ir[ridx], f]
            cp_rr = obs.L[dd.iu[ridx], f] * lam
            cp_br = obsb.L[dd.ir[ridx], f] * lam
            if 0.0 in (pr_rr, pr_br, cp_rr, cp_br):
                continue
            rs_ref, rsb_ref = dd.rs[dd.iu[ridx], :3], dd.rsb[dd.ir[ridx], :3]
            sd_ref = ((cp_rr - cp_br) - (pr_rr - pr_br)) / lam   # gauge value
            if AM(ref, f) not in seen_am:
                val.insert(AM(ref, f), float(sd_ref)); seen_am.add(AM(ref, f))
            if (ref, f) not in pinned:
                graph.addPriorDouble(AM(ref, f), sd_ref,
                                     gtsam.noiseModel.Isotropic.Sigma(1, 0.5))
                pinned.add((ref, f))
            for k in ks:
                js = int(dd.sat[k])
                if k == ridx:
                    continue
                pr_tr, pr_tb = obs.P[dd.iu[k], f], obsb.P[dd.ir[k], f]
                cp_tr = obs.L[dd.iu[k], f] * lam
                cp_tb = obsb.L[dd.ir[k], f] * lam
                if 0.0 in (pr_tr, pr_tb, cp_tr, cp_tb):
                    continue
                rs_j, rsb_j = dd.rs[dd.iu[k], :3], dd.rsb[dd.ir[k], :3]
                w = 1.0 / max(np.sin(min(dd.el[k], dd.el[ridx])), 0.1)
                graph.add(gtsam.DoubleDifferencePseudorangeFactor(
                    X, pr_rr, pr_br, pr_tr, pr_tb,
                    gtsam.Point3(*rs_ref), gtsam.Point3(*rs_j),
                    gtsam.Point3(*rsb_ref), gtsam.Point3(*rsb_j),
                    gtsam.Point3(*rb), gtsam.noiseModel.Isotropic.Sigma(1, 0.3 * w)))
                sd_tgt = ((cp_tr - cp_tb) - (pr_tr - pr_tb)) / lam
                if AM(js, f) not in seen_am:
                    val.insert(AM(js, f), float(sd_tgt)); seen_am.add(AM(js, f))
                graph.add(gtsam.DoubleDifferenceCarrierPhaseFactor(
                    X, AM(ref, f), AM(js, f), cp_rr, cp_br, cp_tr, cp_tb,
                    gtsam.Point3(*rs_ref), gtsam.Point3(*rs_j),
                    gtsam.Point3(*rsb_ref), gtsam.Point3(*rsb_j),
                    gtsam.Point3(*rb), lam,
                    gtsam.noiseModel.Isotropic.Sigma(1, 0.01 * w)))
    nfac += graph.size()
    isam.update(graph, val)

    res = isam.calculateEstimate()
    xh = np.array(res.atPoint3(X))
    enu = gn.ecef2enu(pos_ref, xh - xyz_ref)
    if ei % 10 == 0 or ei == len(frames) - 1:
        print(f'ep{ei:3d} float 2D={np.hypot(enu[0], enu[1]):.3f} '
              f'3D={np.linalg.norm(xh - xyz_ref):.3f} m')
print(f'\ngraph: {nfac} factors, {len(seen_am)} ambiguities')
res = isam.calculateEstimate()

ep  0 float 2D=0.088 3D=0.089 m
ep 10 float 2D=0.131 3D=0.258 m
ep 20 float 2D=0.161 3D=0.251 m
ep 30 float 2D=0.161 3D=0.252 m
ep 40 float 2D=0.167 3D=0.289 m


ep 50 float 2D=0.159 3D=0.289 m


ep 58 float 2D=0.155 3D=0.262 m

graph: 3393 factors, 34 ambiguities


## 5. Integer ambiguity resolution (explicit)

Now we resolve the integers on the final float estimate. The bridge from GTSAM to
cssrlib's LAMBDA (`resamb_lambda`) is unfolded here step by step instead of hidden
in a helper:

1. Write the float SD ambiguities into the cssrlib `nav` state (`nav.x`).
2. Write the **position covariance** (`isam.marginalCovariance(X)`) into `nav.P`.
3. Write the **ambiguity covariance**. The full position+ambiguity joint marginal is
   numerically ill-conditioned (Point3 mixed with many correlated cycle-scale
   ambiguities -> NaN), so it is assembled from the *ambiguity-only* joint (stable)
   plus *pairwise* (position, ambiguity) cross terms.
4. Guard any non-finite entry, then call `resamb_lambda` (LAMBDA + `ddidx`
   single-difference mapping + ratio test). Only the AR *algorithm* is used -- not
   cssrlib's EKF.

In [6]:
nav = rtk.nav

# (1) float ambiguities -> nav.x ; reset the rest of the state
nav.x[nav.na:] = 0.0
nav.P[:, :] = 0.0
nav.vsat[:, :] = 0
nav.x[0:3] = np.array(res.atPoint3(X))

# the SD ambiguities present in the graph (use the last epoch's sat/el)
obs, obsb, dd = frames[-1]
amb = [(int(s), f) for s in dd.sat for f in range(nf)
       if sat2prn(int(s))[0] in SYSS and AM(int(s), f) in seen_am
       and res.exists(AM(int(s), f))]
el_now = {int(s): dd.el[i] for i, s in enumerate(dd.sat)}
for (s, f) in amb:
    j = rtk.IB(s, f, nav.na)
    nav.x[j] = res.atDouble(AM(s, f))
    nav.vsat[s - 1, f] = 1
    nav.el[s - 1] = el_now[s]
print(f'{len(amb)} float ambiguities written into nav.x')

# (2) position covariance (ISAM2 QR -> robust marginal)
nav.P[0:3, 0:3] = isam.marginalCovariance(X)

# (3a) ambiguity-only joint marginal (stable; full X+amb joint would be NaN)
kv = gtsam.KeyVector([AM(s, f) for (s, f) in amb])
jm = isam.jointMarginalCovariance(kv)
for (s, f) in amb:
    j = rtk.IB(s, f, nav.na)
    nav.P[j, j] = jm.at(AM(s, f), AM(s, f))[0, 0]
    # (3b) pairwise (position, ambiguity) cross-covariance
    pxn = isam.jointMarginalCovariance(
        gtsam.KeyVector([X, AM(s, f)])).at(X, AM(s, f))[:, 0]
    nav.P[0:3, j] = pxn; nav.P[j, 0:3] = pxn
# (3c) ambiguity-ambiguity cross terms from the ambiguity-only joint
for a in range(len(amb)):
    s1, f1 = amb[a]; j1 = rtk.IB(s1, f1, nav.na)
    for b in range(a + 1, len(amb)):
        s2, f2 = amb[b]; j2 = rtk.IB(s2, f2, nav.na)
        c = jm.at(AM(s1, f1), AM(s2, f2))[0, 0]
        nav.P[j1, j2] = c; nav.P[j2, j1] = c

# (4) guard non-finite entries, then run LAMBDA
bad = ~np.isfinite(nav.P)
if bad.any():
    nav.P[bad] = 0.0
    d = np.where(np.diag(bad))[0]; nav.P[d, d] = 1e10
nav.elmaskar = np.deg2rad(15.0)
sat_ar = np.array(sorted({s for (s, f) in amb}))
nb, _ = rtk.resamb_lambda(sat_ar, nav.parmode, nav.par_P0)
fixed_xyz = np.array(nav.xa[0:3]) if nb > 0 else None
print(f'resamb_lambda: {nb} SD ambiguities fixed '
      f'(ratio test threshold nav.thresar={nav.thresar})')

32 float ambiguities written into nav.x
resamb_lambda: 28 SD ambiguities fixed (ratio test threshold nav.thresar=2.0)


## 6. Results

With ~23 satellites on two frequencies and a short baseline, RTK fixes
**instantaneously** and the fixed solution agrees with the surveyed marker at the
**mm-cm** level (the float solution is already dm-level; the integer fix tightens it).

In [7]:
xf = np.array(res.atPoint3(X))
enu_f = gn.ecef2enu(pos_ref, xf - xyz_ref)
print(f'float: 2D={np.hypot(enu_f[0], enu_f[1]):.3f} '
      f'3D={np.linalg.norm(xf - xyz_ref):.3f} m')
if fixed_xyz is not None:
    enu_x = gn.ecef2enu(pos_ref, fixed_xyz - xyz_ref)
    print(f'FIX  : 2D={np.hypot(enu_x[0], enu_x[1]):.3f} '
          f'3D={np.linalg.norm(fixed_xyz - xyz_ref):.3f} m  ({nb} SD ambiguities)')

float: 2D=0.155 3D=0.262 m
FIX  : 2D=0.007 3D=0.015 m  (28 SD ambiguities)
